In [44]:
import os
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

openai_api_key = os.getenv("OPENAI_API_KEY")

# Load website
loader = WebBaseLoader(web_paths=['https://medigence.com/hospital/seven-hills', 
                                  'https://medigence.com/doctors/all', 
                                  'https://medigence.com/clinics/all', 
                                  'https://medigence.com/packages'])
docs = loader.load()

# Split
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)
splits = text_splitter.split_documents(docs)

# Create vector store & persist
vectorstore = Chroma.from_documents(
    documents=splits,
    embedding=OpenAIEmbeddings(),
    persist_directory="chroma_db"
)

print("Vector store created successfully!")

Vector store created successfully!


### WebScrapping

In [1]:
import os
from langchain_community.document_loaders import WebBaseLoader

openai_api_key = os.getenv("OPENAI_API_KEY")

loader = WebBaseLoader(web_paths=['https://www.kimshospitals.com/investors/'])

docs = loader.load()

print(docs)

USER_AGENT environment variable not set, consider setting it to identify your requests.


[Document(metadata={'source': 'https://www.kimshospitals.com/investors/', 'title': 'Investors | KIMS Hospitals', 'description': 'Kims Hospitals', 'language': 'en'}, page_content="\n\n\nInvestors | KIMS Hospitals\n\n\n\n          Dial 040-44885000\n        Or\n        \n          Press Here to Book an Appointment\n         \n            Location:\n           \n                Secunderabad\n                  Telangana    \n            KIMS Secunderabad\n          \n            KIMS Kondapur - Hyderabad\n          \n            KIMS Gachibowli - Hyderabad\n          \n            KIMS-Sunshine - Begumpet\n          \n            KIMS Kompally\n          Andhra Pradesh    \n            KIMS Nellore\n          \n            KIMS Srikakulam\n          \n            KIMS Rajahmundry\n          \n            KIMS Ongole\n          \n            KIMS-Saveera Anantapur\n          \n            KIMS Kurnool\n          \n            KIMS-ICON Vizag - Sheelanagar\n          \n            KIMS Vizag

### Splitting

In [14]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=20)
splits = text_splitter.split_documents(docs)

In [24]:
print(len(splits))

498


### Embedding and Store to Vector DB

In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

vectorstore = Chroma.from_documents(
                                    documents = splits, 
                                    embedding = OpenAIEmbeddings())

In [28]:
vectorstore._collection.count()

498

### Retriever

In [29]:
retriever = vectorstore.as_retriever()

### Augmentation

In [32]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("""
You are a Medical Assistant Chatbot.

Use the following pieces of retrieved context to answer the question. 
Use three sentences maximum and keep the answer concise.
If you don't know the answer, just say that "I don't have enough information to answer that" and 
provide Hospital Contact Number for Query and timings to call.

Context:
{context}

Question:
{question}

Answer:
""")

### LLM Steup

In [ ]:
from langchain_openai import ChatOpenAI
llm = ChatOpenAI()

### RAG Chain Setup

In [33]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

In [34]:
def format_docs(docs):
  return "\n".join(doc.page_content for doc in docs)

In [35]:
rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

1) retriever = gives splits that are relevant.  
2) format_docs = Extracts only text and Joins them into one big context string.  
3) RunnablePassthrough = Whatever user question comes → pass directly.  
4) StrOutputParser = LLM returns structured response object. This converts it into plain string output. Without it, you get a complex object.
5) rag_chain = runnable object. It is a LangChain Runnable sequence that defines:  
           - What happens first  
           - What happens next  
           - What happens after that.  
    It does NOT execute immediately.  
    It just defines the pipeline.  
6) | = is NOT normal Python pipe. In LangChain, it is operator overloading. It means:  
        "Take output of left side and pass it as input to right side"

Question  
   ↓  
retriever(question)  
   ↓  
documents  
   ↓  
format_docs(documents)  
   ↓  
formatted string  

then |prompt means -- Take dictionary Insert into prompt template.  

then |llm means -- Send formatted prompt to LLM.  

then |StrOutputParser() means -- Convert model output to string.  

{
  "context": retriever | format_docs,
  "question": RunnablePassthrough()
}

This is called a Runnable Map.  
It means:  
Take ONE input and produce a dictionary output.

RunnableSequence(  
    RunnableMap(...),  
    prompt,  
    llm,  
    StrOutputParser()  
)

In [38]:
rag_chain.invoke('Where is the Hospital Located?')

'The Seven Hills Hospital is located on Marol Maroshi Rd, Mahavir Nagar, Andheri in India.'

In [42]:
rag_chain.invoke('Can I do Internship here?')

"I don't have enough information to answer that. Please contact the hospital at [Hospital Contact Number] for queries and timings to call."